# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZubairQazzi/flyrank-ml-internship-zubair/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [10]:
from google.colab import userdata
import duckdb
import pandas as pd
import numpy as np

hf_token = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"""
    CREATE SECRET hf_secret (
        TYPE huggingface,
        TOKEN '{hf_token}'
    )
    """
)

march_path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/*.parquet"
)

content_path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "dim_content.parquet"
)

print("Setup complete")

Setup complete


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — The Anatomy of Growing Content

The paper reports that growing pages tend to be longer, younger, and slightly better positioned than declining pages. It also describes this as an observational comparison rather than causal proof.

**Methodology question:**  
How exactly was the growing-versus-declining label defined, and were word count, age, and position measured strictly before the trend window or during the same period?

**Why I would ask this:**  
If the explanatory fields overlap with the period used to define growth or decline, part of the observed relationship may reflect information from the outcome window. I would want the timeline to confirm that the features were knowable before the label period.

### Finding 2 — AI Model Performance

The paper reports that OpenAI and Gemini each lead in different age-controlled cohorts, so the result is treated as exploratory rather than proof that one provider universally performs better.

**Methodology question:**  
After controlling for content age, are the OpenAI and Gemini cohorts also comparable by client, topic, editorial process, and publication period?

**Why I would ask this:**  
Age control removes one important confounder, but differences in topic mix, client behavior, editing standards, or rollout timing could still explain some of the observed differences. I would therefore interpret the result as directional evidence rather than a universal provider ranking.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### Validation design: before vs after

I will re-run the same Random Forest model from Week 5 using the same five pre-decision features and the same fixed random seed.

**Before:** a standard random row split. This can place pages from the same client in both training and test sets, which may make performance look better because pages from one client can share hidden site-level patterns.

**After:** a grouped split by `client_hash_id`. Every client appears in either training or test, never both. This better measures whether the model generalizes to a client it has not seen before.

I will compare both splits using the same metrics: test base rate, precision@10, precision@20, and ROC-AUC. The model and features stay unchanged so the only important change is the validation design.

In [12]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

RANDOM_STATE = 42

model_query = f"""
WITH daily_windows AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-21'
                 AND gsc_data_available IS TRUE
                THEN gsc_impressions
                ELSE 0
            END
        ) AS past_impressions,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-21'
                 AND gsc_data_available IS TRUE
                THEN gsc_clicks
                ELSE 0
            END
        ) AS past_clicks,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-21'
                 AND gsc_data_available IS TRUE
                THEN gsc_sum_position
                ELSE 0
            END
        ) AS past_sum_position,

        COUNT(*) FILTER (
            WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-21'
              AND gsc_data_available IS TRUE
        ) AS gsc_observed_days,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-22' AND DATE '2026-03-31'
                 AND gsc_data_available IS TRUE
                THEN gsc_impressions
                ELSE 0
            END
        ) AS outcome_impressions,

        COUNT(*) FILTER (
            WHERE report_date BETWEEN DATE '2026-03-22' AND DATE '2026-03-31'
              AND gsc_data_available IS TRUE
        ) AS outcome_days

    FROM read_parquet('{march_path}')
    GROUP BY client_hash_id, content_hash_id
),

content_status AS (
    SELECT
        client_hash_id,
        content_hash_id,
        is_published,
        is_deleted
    FROM read_parquet('{content_path}')
)

SELECT
    d.client_hash_id,
    d.content_hash_id,
    d.past_impressions,
    d.past_clicks,

    ROUND(
        100.0 * d.past_clicks / NULLIF(d.past_impressions, 0),
        4
    ) AS past_ctr,

    ROUND(
        1.0 * d.past_sum_position / NULLIF(d.past_impressions, 0),
        4
    ) AS past_avg_position,

    d.gsc_observed_days,

    CASE
        WHEN
            (1.0 * d.outcome_impressions / d.outcome_days)
            <
            0.80 * (
                1.0 * d.past_impressions / d.gsc_observed_days
            )
        THEN 1
        ELSE 0
    END AS future_decline

FROM daily_windows d
INNER JOIN content_status c
    ON d.client_hash_id = c.client_hash_id
   AND d.content_hash_id = c.content_hash_id

WHERE
    c.is_published IS TRUE
    AND COALESCE(c.is_deleted, FALSE) IS FALSE
    AND d.gsc_observed_days >= 7
    AND d.outcome_days >= 5
    AND d.past_impressions >= 100
"""

model_frame = con.sql(model_query).df()

features = [
    "past_impressions",
    "past_clicks",
    "past_ctr",
    "past_avg_position",
    "gsc_observed_days",
]

X = model_frame[features].astype(float)
y = model_frame["future_decline"].astype(int)
groups = model_frame["client_hash_id"]


def precision_at_k(scores, labels, k):
    scores = np.asarray(scores)
    labels = np.asarray(labels)

    order = np.argsort(-scores)
    k = min(k, len(order))

    return labels[order[:k]].mean()


def make_rf():
    return RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=20,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )


# ---------------------------------------
# BEFORE: ordinary random row split
# ---------------------------------------

all_idx = np.arange(len(model_frame))

random_train_idx, random_test_idx = train_test_split(
    all_idx,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

rf_random = make_rf()

rf_random.fit(
    X.iloc[random_train_idx],
    y.iloc[random_train_idx],
)

random_scores = rf_random.predict_proba(
    X.iloc[random_test_idx]
)[:, 1]

random_test_y = y.iloc[random_test_idx]

random_train_clients = set(
    model_frame.iloc[random_train_idx]["client_hash_id"]
)

random_test_clients = set(
    model_frame.iloc[random_test_idx]["client_hash_id"]
)

random_overlap = len(
    random_train_clients & random_test_clients
)


# ---------------------------------------
# AFTER: grouped by client
# ---------------------------------------

group_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

group_train_idx, group_test_idx = next(
    group_splitter.split(X, y, groups=groups)
)

rf_grouped = make_rf()

rf_grouped.fit(
    X.iloc[group_train_idx],
    y.iloc[group_train_idx],
)

group_scores = rf_grouped.predict_proba(
    X.iloc[group_test_idx]
)[:, 1]

group_test_y = y.iloc[group_test_idx]

group_train_clients = set(
    model_frame.iloc[group_train_idx]["client_hash_id"]
)

group_test_clients = set(
    model_frame.iloc[group_test_idx]["client_hash_id"]
)

group_overlap = len(
    group_train_clients & group_test_clients
)


# ---------------------------------------
# Before / after comparison
# ---------------------------------------

validation_comparison = pd.DataFrame(
    [
        {
            "split": "Random row split (before)",
            "test_rows": len(random_test_idx),
            "test_base_rate": random_test_y.mean(),
            "client_overlap": random_overlap,
            "precision@10": precision_at_k(
                random_scores, random_test_y, 10
            ),
            "precision@20": precision_at_k(
                random_scores, random_test_y, 20
            ),
            "roc_auc": roc_auc_score(
                random_test_y, random_scores
            ),
        },
        {
            "split": "Grouped client split (after)",
            "test_rows": len(group_test_idx),
            "test_base_rate": group_test_y.mean(),
            "client_overlap": group_overlap,
            "precision@10": precision_at_k(
                group_scores, group_test_y, 10
            ),
            "precision@20": precision_at_k(
                group_scores, group_test_y, 20
            ),
            "roc_auc": roc_auc_score(
                group_test_y, group_scores
            ),
        },
    ]
)

validation_comparison.round(4)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,split,test_rows,test_base_rate,client_overlap,precision@10,precision@20,roc_auc
0,Random row split (before),17194,0.3249,33,0.6,0.7,0.6351
1,Grouped client split (after),4247,0.4196,0,0.5,0.5,0.5543


Before/after validation result
The random row split allowed 33 clients to appear in both training and test data, while the grouped split reduced client overlap to zero.

Precision@10 fell from 0.60 under the random split to 0.50 under the grouped client split, while precision@20 fell from 0.70 to 0.50. ROC-AUC also fell from 0.6351 to 0.5543. The grouped test population had a higher observed base rate, 41.96% versus 32.49%.

This does not mean the grouped model became worse. The grouped evaluation asks the harder and more realistic question of whether the model generalizes to clients it has never seen before. I therefore treat the grouped-client result as the more trustworthy decision-support estimate.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Leakage audit

I will audit the five final Week-5 model features against the Week-6 leakage checklist.

The prediction moment is the end of the feature window on **2026-03-21**. The label uses the later outcome window, **2026-03-22 through 2026-03-31**.

A feature is safe only if it is fully knowable by the prediction moment and is not derived from the future label, an outcome-window field, an identifier, or an existing product decision flag.

The five model features are:

1. `past_impressions`
2. `past_clicks`
3. `past_ctr`
4. `past_avg_position`
5. `gsc_observed_days`

All five use only the March 1–21 feature window. The `future_decline` label uses the later March 22–31 outcome window and is not a model input.

Client and content IDs are used only for grouping and audit context. Product flags and existing-system scores are not model features.

In [13]:
leakage_audit = pd.DataFrame(
    [
        {
            "feature": "past_impressions",
            "source_window": "2026-03-01 to 2026-03-21",
            "known_at_decision_time": True,
            "label_derived": False,
            "product_flag": False,
            "verdict": "SAFE",
        },
        {
            "feature": "past_clicks",
            "source_window": "2026-03-01 to 2026-03-21",
            "known_at_decision_time": True,
            "label_derived": False,
            "product_flag": False,
            "verdict": "SAFE",
        },
        {
            "feature": "past_ctr",
            "source_window": "2026-03-01 to 2026-03-21",
            "known_at_decision_time": True,
            "label_derived": False,
            "product_flag": False,
            "verdict": "SAFE",
        },
        {
            "feature": "past_avg_position",
            "source_window": "2026-03-01 to 2026-03-21",
            "known_at_decision_time": True,
            "label_derived": False,
            "product_flag": False,
            "verdict": "SAFE",
        },
        {
            "feature": "gsc_observed_days",
            "source_window": "2026-03-01 to 2026-03-21",
            "known_at_decision_time": True,
            "label_derived": False,
            "product_flag": False,
            "verdict": "SAFE",
        },
    ]
)

forbidden_model_inputs = {
    "future_decline",
    "outcome_impressions",
    "outcome_days",
    "client_hash_id",
    "content_hash_id",
}

actual_model_inputs = set(features)

forbidden_found = sorted(
    actual_model_inputs & forbidden_model_inputs
)

print("Prediction moment: 2026-03-21")
print("Label window: 2026-03-22 to 2026-03-31")
print("Model inputs:", features)
print("Forbidden inputs found:", forbidden_found)
print(
    "Leakage audit passed:",
    len(forbidden_found) == 0
    and (leakage_audit["verdict"] == "SAFE").all()
)

leakage_audit

Prediction moment: 2026-03-21
Label window: 2026-03-22 to 2026-03-31
Model inputs: ['past_impressions', 'past_clicks', 'past_ctr', 'past_avg_position', 'gsc_observed_days']
Forbidden inputs found: []
Leakage audit passed: True


,feature,source_window,known_at_decision_time,label_derived,product_flag,verdict
0,past_impressions,2026-03-01 to 2026-03-21,True,False,False,SAFE
1,past_clicks,2026-03-01 to 2026-03-21,True,False,False,SAFE
2,past_ctr,2026-03-01 to 2026-03-21,True,False,False,SAFE
3,past_avg_position,2026-03-01 to 2026-03-21,True,False,False,SAFE
4,gsc_observed_days,2026-03-01 to 2026-03-21,True,False,False,SAFE


### Real failure examples

I also inspect concrete errors from the grouped-client evaluation rather than relying only on aggregate metrics.

A high-scoring false positive is a page the model ranked as relatively risky but that did not show the later decline proxy. A low-scoring missed decline is a page that later declined even though the model assigned it relatively low risk.

These cases show where the five-feature model is limited and why its output should be used for decision support rather than automatic action.

In [14]:
error_frame = model_frame.iloc[group_test_idx].copy()

error_frame["model_score"] = group_scores
error_frame["actual_decline"] = group_test_y.to_numpy()

false_positives = (
    error_frame[error_frame["actual_decline"] == 0]
    .sort_values("model_score", ascending=False)
    .head(3)
    [
        [
            "content_hash_id",
            "past_impressions",
            "past_clicks",
            "past_ctr",
            "past_avg_position",
            "gsc_observed_days",
            "model_score",
            "actual_decline",
        ]
    ]
)

missed_declines = (
    error_frame[error_frame["actual_decline"] == 1]
    .sort_values("model_score", ascending=True)
    .head(3)
    [
        [
            "content_hash_id",
            "past_impressions",
            "past_clicks",
            "past_ctr",
            "past_avg_position",
            "gsc_observed_days",
            "model_score",
            "actual_decline",
        ]
    ]
)

print("Three high-scoring false positives:")
display(false_positives.round(4))

print("\nThree low-scoring missed declines:")
display(missed_declines.round(4))

Three high-scoring false positives:


,content_hash_id,past_impressions,past_clicks,past_ctr,past_avg_position,gsc_observed_days,model_score,actual_decline
37133,content_66cb7a06190f8b36,908.0,0.0,0.0,0.3183,17,0.7307,0
80397,content_ee18e4561fdebed4,863.0,0.0,0.0,0.4647,18,0.7142,0
37118,content_ff00e27495974042,273.0,0.0,0.0,0.4469,20,0.7089,0



Three low-scoring missed declines:


,content_hash_id,past_impressions,past_clicks,past_ctr,past_avg_position,gsc_observed_days,model_score,actual_decline
75248,content_cb0a771e574ebe26,18910.0,90.0,0.4759,2.9655,21,0.1529,1
53288,content_4f6b6f419514c27b,8934.0,54.0,0.6044,4.0343,21,0.1627,1
10736,content_5ebc94f67db6f51c,16008.0,416.0,2.5987,2.4672,21,0.1642,1


**Population-selection limitation:**  
The analysis requires at least five usable GSC days in the March 22–31 outcome window (`outcome_days >= 5`). This means eligibility is partly determined using information from the outcome period. I do not use that information as a model feature, but I disclose it because the evaluated population excludes pages without sufficient later observations. The reported metrics therefore apply to this observed subset rather than every content item.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Claim rewrite

**My earlier Week-5 claim:**

> Both learned models clearly improved over the Week 4 rule baseline.

This wording is too broad because the measured improvement depends on the validation design. The random row split produced more optimistic results and allowed client overlap, while the grouped-client split evaluates generalization to unseen clients.

**Rewritten public-safe claim:**

> On the grouped-client holdout used in this audit, the learned models measured higher top-of-queue precision than the Week-4 rule baseline. The result is directional evidence that the five pre-decision features provide useful ranking signal, but performance remains modest and varies with the validation design. The model should therefore be used for decision support rather than automatic refresh decisions.

The failure examples reinforce this limitation. Some pages with zero observed clicks received high risk scores without later declining, while several pages with healthy historical visibility still declined despite receiving low model scores. These errors show that the available historical features do not capture every factor associated with future search-performance changes.

In [15]:
claim_audit = pd.DataFrame(
    [
        {
            "version": "Earlier claim",
            "claim": (
                "Both learned models clearly improved over the "
                "Week 4 rule baseline."
            ),
            "status": "TOO BROAD",
        },
        {
            "version": "Audited claim",
            "claim": (
                "On the grouped-client holdout, the learned models "
                "measured higher top-of-queue precision than the "
                "Week-4 baseline, but performance remains modest "
                "and should be treated as decision-support evidence."
            ),
            "status": "PUBLIC-SAFE",
        },
    ]
)

claim_audit

,version,claim,status
0,Earlier claim,Both learned models clearly improved over the ...,TOO BROAD
1,Audited claim,"On the grouped-client holdout, the learned mod...",PUBLIC-SAFE


- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.